# External dataset audit

Count carrot, cabbage and potato reference parcels or classified pixels using local files. Run from the repo root or notebooks folder using the project environment. No Earth Engine access is required. All data paths can be changed below.

- **CropHarvest** uses the pinned [Zenodo label release](https://zenodo.org/records/10251170). The full audit gives 552 distinct screened parcels with an interior 10 m pixel, including 221 JECAM parcels. Imagery and window quality are evaluated separately.
- **EuroCrops v2.0** uses [annual GSA GeoParquet files](https://jeodpp.jrc.ec.europa.eu/ftp/public/JRC-OpenData/DRLL/EuroCropsV2/) named `<NUTS>_<YEAR>.parquet` and the [official mapping CSV](https://github.com/Martincccc/EuroCropsV2/blob/main/data/cropcodemapping/eurocrops.csv). Counts are distinct parcel IDs per region and year. Review the helper's exact translated-name aliases before treating a zero as absence.
- **USDA CDL** uses [official classified raster codes](https://developers.google.com/earth-engine/datasets/catalog/USDA_NASS_CDL). Counts are classified pixels per raster. Field boundaries are needed to count parcels.

In [ ]:
from pathlib import Path

from crop_pipeline.sources.dataset_audit import (
    audit_cropharvest, audit_eurocrops, audit_usda_cdl,
)

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
assert (root / "pyproject.toml").is_file(), "Start from the repository root or notebooks folder"
data = root / "data/external"
labels = data / "cropharvest/labels.geojson"
eurocrops = data / "eurocrops_v2"
mapping = eurocrops / "eurocrops.csv"
usda = data / "usda_cdl"

## CropHarvest

In [ ]:
if labels.is_file():
    total, counts, by_source = audit_cropharvest(labels, data / "dataset_audit/cropharvest")
    print(f"Full release: {total:,} label records")
    print(counts.to_string())
    print(by_source.to_string(index=False))
else:
    print(f"Place the pinned CropHarvest labels.geojson at {labels}")

## EuroCrops v2.0

In [ ]:
files = sorted(eurocrops.rglob("*.parquet"))
if mapping.is_file() and files:
    counts = audit_eurocrops(files, mapping)
    if counts.empty:
        print("No annual GSA files found. Expected filenames such as PT_2021.parquet.")
    else:
        print(counts.pivot(index=["region", "year"], columns="crop", values="declared_parcels").to_string())
else:
    print(f"Place annual GeoParquets in {eurocrops} and the mapping CSV at {mapping}")

## USDA Cropland Data Layer

In [ ]:
files = sorted(path for path in usda.rglob("*") if path.suffix.lower() in {".tif", ".tiff"})
if files:
    for path in files:
        print(path.relative_to(usda))
        print(audit_usda_cdl(path).to_string())
else:
    print(f"Place USDA CDL GeoTIFFs in {usda}")